In [0]:
%sql

CREATE OR REPLACE VIEW
ask_sarthak.analytics.vw_anomaly_transaction_evidence
AS

WITH alert_documents AS (
    SELECT DISTINCT
        i.company_code,
        f.fiscal_year,
        f.document_number
    FROM ask_sarthak.analytics.vw_finance_investigation i
    JOIN ask_sarthak.gold.fact_gl_transactions f
        ON i.company_code = f.company_code
       AND i.vendor_id = f.vendor_id
       AND i.expense_month =
           CAST(DATE_TRUNC('MONTH', f.posting_date) AS DATE)
    JOIN ask_sarthak.gold.dim_gl_account a
        ON f.gl_account = a.gl_account
    WHERE a.account_type = 'EXPENSE'
),

transaction_evidence AS (
    SELECT
        i.company_code,
        i.vendor_id,
        i.vendor_name,
        i.expense_month,
        i.risk_level,
        i.monthly_expense,
        i.historical_avg,
        i.percentage_increase,
        i.z_score,
        i.reconciliation_status,

        f.fiscal_year,
        f.document_number,
        f.line_item,
        f.posting_date,
        f.gl_account,
        a.account_name,
        a.account_type,
        f.debit_amount,
        f.credit_amount,

        f.debit_amount - f.credit_amount
            AS net_amount

    FROM alert_documents d

    JOIN ask_sarthak.gold.fact_gl_transactions f
        ON d.company_code = f.company_code
       AND d.fiscal_year = f.fiscal_year
       AND d.document_number = f.document_number

    JOIN ask_sarthak.analytics.vw_finance_investigation i
        ON d.company_code = i.company_code
       AND f.vendor_id = i.vendor_id
       AND i.expense_month =
           CAST(DATE_TRUNC('MONTH', f.posting_date) AS DATE)

    JOIN ask_sarthak.gold.dim_gl_account a
        ON f.gl_account = a.gl_account
)

SELECT *
FROM transaction_evidence;


In [0]:
%sql
SELECT
    company_code,
    vendor_id,
    expense_month,
    document_number,
    line_item,
    posting_date,
    gl_account,
    account_name,
    account_type,
    debit_amount,
    credit_amount,
    net_amount
FROM ask_sarthak.analytics.vw_anomaly_transaction_evidence
WHERE company_code = '3000'
  AND vendor_id = 'V0153'
  AND expense_month = DATE '2026-10-01'
ORDER BY document_number, line_item;